<a href="https://colab.research.google.com/github/Kaiking28/ECON3916-Statistical-Machine-Learning/blob/main/PHASE%201%20Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [ ]:
np.random.seed(42)
zeros = np.zeros(100)
tips = np.random.exponential(scale=5.0, size=150)
driver_tips = np.concatenate([zeros, tips])

plt.hist(driver_tips[driver_tips > 0])
plt.title('Driver Tip Distribution')
plt.xlabel('Tip ($)')
plt.show()

In [ ]:
np.random.seed(42)
N_BOOT = 10000
n = len(driver_tips)
boot_medians = np.empty(N_BOOT)

for i in range(N_BOOT):
    resample = driver_tips[np.random.randint(0, n, size=n)]
    boot_medians[i] = np.median(resample)

ci_lower = np.percentile(boot_medians, 2.5)
ci_upper = np.percentile(boot_medians, 97.5)
observed_median = np.median(driver_tips)

print('Observed Median:', observed_median)
print('95% CI:', ci_lower, 'to', ci_upper)
print('Lower distance:', observed_median - ci_lower)
print('Upper distance:', ci_upper - observed_median)

plt.hist(boot_medians)
plt.title('Bootstrap Distribution of Median')
plt.xlabel('Bootstrap Median ($)')
plt.show()

**Discussion:** The bootstrap CI is asymmetric so the upper bound is further from the median than the lower bound. This reflects the data's topology: 40% of tips are exactly $0, creating a hard floor that compresses the lower tail, while the exponential right-skew stretches the upper tail. A parametric interval assumes symmetry by construction and would miss this entirely.

In [ ]:
np.random.seed(42)
N = 500

control = np.random.normal(loc=35, scale=5, size=N)
treatment = np.random.lognormal(mean=3.4, sigma=0.4, size=N)
obs_diff = control.mean() - treatment.mean()

print('Control mean:', control.mean(), 'std:', control.std())
print('Treatment mean:', treatment.mean(), 'std:', treatment.std())
print('Observed difference (C - T):', obs_diff)

fig, axes = plt.subplots(1, 2)
axes[0].hist(control)
axes[0].set_title('Control')
axes[1].hist(treatment)
axes[1].set_title('Treatment')
plt.show()

In [ ]:
np.random.seed(42)
N_PERM = 5000
combined = np.concatenate([control, treatment])
perm_diffs = np.empty(N_PERM)

for i in range(N_PERM):
    shuffled = np.random.permutation(combined)
    perm_diffs[i] = shuffled[:N].mean() - shuffled[N:].mean()

p_value = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))

print('Observed difference:', obs_diff)
print('Empirical p-value:', p_value)
print('Reject H0' if p_value < 0.05 else 'Fail to reject H0')

plt.hist(perm_diffs)
plt.title('Permutation Null Distribution')
plt.xlabel('Simulated Difference in Means')
plt.show()

In [ ]:
df = pd.read_csv('swiftcart_loyalty.csv')
print(df.head())
print(df.groupby('subscriber')[['pre_spend','account_age','support_tickets','post_spend']].mean())

mean_sub = df.loc[df['subscriber'] == 1, 'post_spend'].mean()
mean_nonsub = df.loc[df['subscriber'] == 0, 'post_spend'].mean()
naive_sdo = mean_sub - mean_nonsub

print('Subscriber mean:', mean_sub)
print('Non-subscriber mean:', mean_nonsub)
print('Naive SDO:', naive_sdo)

In [ ]:
COVARIATES = ['pre_spend', 'account_age', 'support_tickets']

X = StandardScaler().fit_transform(df[COVARIATES])
D = df['subscriber'].values

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X, D)
df['propensity_score'] = lr.predict_proba(X)[:, 1]

treated = df[df.subscriber == 1].reset_index(drop=True)
control_pool = df[df.subscriber == 0].reset_index(drop=True)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control_pool[['propensity_score']])
_, indices = nn.kneighbors(treated[['propensity_score']])
matched_control = control_pool.iloc[indices.flatten()].reset_index(drop=True)

att = (treated['post_spend'].values - matched_control['post_spend'].values).mean()

print('Naive SDO:', naive_sdo)
print('PSM ATT:', att)
print('Bias removed:', naive_sdo - att)

**Analysis:** The naive SDO is inflated by selection bias as subscribers are high-spending power users who self-selected into SwiftPass. PSM matches each subscriber to the most similar non-subscriber by propensity score, constructing a valid counterfactual and isolating the true causal effect. The ATT is materially lower than the SDO; doubling the acquisition budget based on the naive SDO would misallocate capital based on selection artefact, not causality.

In [ ]:
def compute_smd(t_df, c_df, features):
    smds = {}
    for f in features:
        pooled_sd = np.sqrt((t_df[f].var() + c_df[f].var()) / 2)
        smds[f] = (t_df[f].mean() - c_df[f].mean()) / pooled_sd
    return smds

smd_before = compute_smd(df[df.subscriber==1][COVARIATES], df[df.subscriber==0][COVARIATES], COVARIATES)
smd_after  = compute_smd(treated[COVARIATES], matched_control[COVARIATES], COVARIATES)

labels = ['Pre-Treatment Spending', 'Account Age', 'Support Tickets']
smd_b = list(smd_before.values())
smd_a = list(smd_after.values())
y_pos = np.arange(len(COVARIATES))

plt.scatter(smd_b, y_pos + 0.1, label='Before')
plt.scatter(smd_a, y_pos - 0.1, label='After')
plt.yticks(y_pos, labels)
plt.axvline(x=0)
plt.axvline(x=0.1, linestyle='--')
plt.axvline(x=-0.1, linestyle='--')
plt.xlabel('Standardized Mean Difference')
plt.title('Love Plot: Covariate Balance Before and After PSM')
plt.legend()
plt.show()

**Love Plot Evaluation:** The points before matching sit far from zero, confirming systematic differences between subscribers and non-subscribers on all covariates. After PSM, all points collapse near zero and within the +-0.10 threshold lines, meaning the matched groups are now statistically indifferentiable on all observable confounders. This is the visual proof that selection bias was successfully mitigated and the ATT is causally valid under the Conditional Independence Assumption.